Task description:

**Case Study: Engine Breakdown Risk – Backend + Analytics + Tableau**

You are part of the BI &amp; Analytics team responsible for monitoring and predicting engine
breakdown risk. Senior management wants a reliable backend data foundation and a clear
Tableau dashboard that explains the drivers of breakdowns and supports proactive decision-
making.
You are given a dataset of historical engine operation records. Your task is to:
1. Design the backend data model and transformations needed to support analysis and
dashboards.
2. Analyze the data and identify patterns related to the target variable breakdown.
3. Propose or build a predictive approach for estimating breakdown risk.
4. Create a Tableau dashboard that communicates insights to senior management. (twbx)
5. Prepare a short presentation summarizing your findings, insights, and
recommendations. (ppt)

## Imports and functions

In [30]:
import pandas as pd
from io import StringIO

In [31]:
def read_input_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    content = "\n".join(
        line.strip('"') for line in content.splitlines()
    )

    df = pd.read_csv(
        StringIO(content),
        sep=","
    )
    return df

## Input table description

| Column | Description | Values / Meaning |
|---|---|---|
| `oph` | Operating hours of the engine | — |
| `pist_m` | Piston material | — |
| `issue_type` | Combustion issue type | `1` = typical issue<br>`2` = atypical issue<br>`3` = non-related<br>`4` = non-symptomatic |
| `bmep` | Brake mean effective pressure (average pressure forcing the pistons down) | — |
| `ng_imp` | Natural gas impurities (nmol) | — |
| `past_dmg` | Engine had past damages | `1` = true<br>`0` = false |
| `resting_analysis_results` | Resting results after operation | `0` = normal<br>`1` = abnormal<br>`2` = critical |
| `rpm_max` | Maximum rotations per minute achieved | — |
| `full_load_issues` | Issues induced due to full load operation | `1` = yes<br>`0` = no |
| `number_up` | Number of unplanned events | — |
| `number_tc` | Number of installed turbochargers | — |
| `op_set_1` | Operational engine setting 1 | — |
| `op_set_2` | Operational engine setting 2 | — |
| `op_set_3` | Operational engine setting 3 | — |
| `breakdown` | Breakdown probability indicator | `0` = less chance of breakdown<br>`1` = more chance of breakdown |

## File read in

In [32]:
FILE_PATH = "./data/BI Fullstack case study dataset.csv"
df = read_input_file(FILE_PATH)
df


,engine_id,timestamp,oph,pist_m,issue_type,bmep,ng_imp,past_dmg,resting_analysis_results,rpm_max,full_load_issues,number_up,number_tc,op_set_1,op_set_2,op_set_3,breakdown
0,ENG001,2021-02-14,5120,2,1,14.8,210,0,0,1485,0.0,1.0,1.0,0.81,0.76,0.88,0.0
1,ENG001,2021-06-22,8450,1,2,17.2,260,1,1,1530,1.0,3.0,2.0,0.84,0.79,0.91,1.0
2,ENG001,2022-01-09,3920,3,1,13.1,150,0,0,1440,0.0,0.0,1.0,0.74,0.78,0.82,0.0
3,ENG001,2022-08-17,10210,2,4,18.4,320,1,2,1585,1.0,4.0,2.0,0.89,0.83,0.94,1.0
4,ENG001,2023-03-11,7210,1,3,12.7,140,0,1,1425,0.0,2.0,1.0,0.72,0.75,0.80,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,ENG050,2023-03-24,6720,3,3,13.2,160,0,1,1445,0.0,2.0,1.0,0.74,0.78,0.82,0.0
497,ENG050,2023-07-11,10280,1,4,17.8,320,1,2,1580,1.0,4.0,2.0,0.88,0.83,0.94,1.0
498,ENG050,2023-11-03,4950,2,1,13.9,170,0,0,1460,0.0,1.0,1.0,0.76,0.79,0.84,0.0
499,ENG050,2024-03-29,9100,3,2,17.0,275,1,1,1510,1.0,3.0,2.0,0.83,0.78,0.90,1.0


## Sanity checks

### Check for unique values

In [33]:
for col in df.columns:
    print(col)
    print(df[col].unique())
    print()

engine_id
<StringArray>
['ENG001', 'ENG002', 'ENG003', 'ENG004', 'ENG005', 'ENG006', 'ENG007',
 'ENG008', 'ENG009', 'ENG010', 'ENG011', 'ENG012', 'ENG013', 'ENG014',
 'ENG015', 'ENG016', 'ENG017', 'ENG018', 'ENG019', 'ENG020', 'ENG021',
 'ENG022', 'ENG023', 'ENG024', 'ENG025', 'ENG026', 'ENG027', 'ENG028',
 'ENG029', 'ENG030', 'ENG031', 'ENG032', 'ENG033', 'ENG034', 'ENG035',
 'ENG036', 'ENG037', 'ENG038', 'ENG039', 'ENG040', 'ENG041', 'ENG042',
 'ENG043', 'ENG044', 'ENG045', 'ENG046', 'ENG047', 'ENG048', 'ENG049',
 'ENG050']
Length: 50, dtype: str

timestamp
<StringArray>
['2021-02-14', '2021-06-22', '2022-01-09', '2022-08-17', '2023-03-11',
 '2023-11-28', '2024-04-19', '2024-07-03', '2024-09-15', '2024-12-06',
 ...
 '2022-02-02', '2022-06-06', '2022-12-02', '2023-10-13', '2024-03-23',
 '2024-09-12', '2022-12-13', '2023-03-24', '2023-07-11', '2024-03-29']
Length: 423, dtype: str

oph
[ 5120  8450  3920 10210  7210 11890  2680 13450  9050  4820 10980  5400
  7800  3150 12840  6550  995

### Check for Nan values

In [34]:
df[df.isna().any(axis=1)]

,engine_id,timestamp,oph,pist_m,issue_type,bmep,ng_imp,past_dmg,resting_analysis_results,rpm_max,full_load_issues,number_up,number_tc,op_set_1,op_set_2,op_set_3,breakdown
440,ENG045,2021-02-21,11990,2,2,18.9,365,1,2,1605,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Check for duplicates

In [35]:
df_non_duplicate = df.drop_duplicates()
len(df_non_duplicate) == len(df)

True

## Copy and Clean

In [36]:
df_copy = df.copy()
df_clean = df_copy.dropna()

### Change type and second check

In [37]:
df_clean['timestamp'] =pd.to_datetime(df_clean['timestamp']).dt.strftime('%Y-%m-%d')

int_columns = [
    'pist_m', 'issue_type', 'past_dmg', 'resting_analysis_results',
    'full_load_issues', 'number_up', 'number_tc', 'breakdown'
]

for col in int_columns:
    df_clean[col] = df_clean[col].astype(int)

print(df_clean.head())

  engine_id   timestamp    oph  pist_m  issue_type  bmep  ng_imp  past_dmg  \
0    ENG001  2021-02-14   5120       2           1  14.8     210         0   
1    ENG001  2021-06-22   8450       1           2  17.2     260         1   
2    ENG001  2022-01-09   3920       3           1  13.1     150         0   
3    ENG001  2022-08-17  10210       2           4  18.4     320         1   
4    ENG001  2023-03-11   7210       1           3  12.7     140         0   

   resting_analysis_results  rpm_max  full_load_issues  number_up  number_tc  \
0                         0     1485                 0          1          1   
1                         1     1530                 1          3          2   
2                         0     1440                 0          0          1   
3                         2     1585                 1          4          2   
4                         1     1425                 0          2          1   

   op_set_1  op_set_2  op_set_3  breakdown  
0    

In [38]:
for col in df_clean.columns:
    print(col)
    print(df_clean[col].unique())
    print()

engine_id
<StringArray>
['ENG001', 'ENG002', 'ENG003', 'ENG004', 'ENG005', 'ENG006', 'ENG007',
 'ENG008', 'ENG009', 'ENG010', 'ENG011', 'ENG012', 'ENG013', 'ENG014',
 'ENG015', 'ENG016', 'ENG017', 'ENG018', 'ENG019', 'ENG020', 'ENG021',
 'ENG022', 'ENG023', 'ENG024', 'ENG025', 'ENG026', 'ENG027', 'ENG028',
 'ENG029', 'ENG030', 'ENG031', 'ENG032', 'ENG033', 'ENG034', 'ENG035',
 'ENG036', 'ENG037', 'ENG038', 'ENG039', 'ENG040', 'ENG041', 'ENG042',
 'ENG043', 'ENG044', 'ENG045', 'ENG046', 'ENG047', 'ENG048', 'ENG049',
 'ENG050']
Length: 50, dtype: str

timestamp
<StringArray>
['2021-02-14', '2021-06-22', '2022-01-09', '2022-08-17', '2023-03-11',
 '2023-11-28', '2024-04-19', '2024-07-03', '2024-09-15', '2024-12-06',
 ...
 '2022-02-02', '2022-06-06', '2022-12-02', '2023-10-13', '2024-03-23',
 '2024-09-12', '2022-12-13', '2023-03-24', '2023-07-11', '2024-03-29']
Length: 423, dtype: str

oph
[ 5120  8450  3920 10210  7210 11890  2680 13450  9050  4820 10980  5400
  7800  3150 12840  6550  995

## Export